In [ ]:
from dotenv import load_dotenv
load_dotenv()


##### Converting script pdf to text

In [ ]:

import pdfplumber
import re

def pdf_text(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"

    return text

##### Extracting charecter names from the script

In [ ]:
def extract_charecters(script,model):

    charecter_prompt="""
# Screenplay Character Analysis System Prompt

## Role
You are a professional screenplay analysis engine, specialized in parsing and extracting structured data from film and television scripts.

## Primary Task
Analyze the provided screenplay text and compile a complete, accurate list of all unique speaking characters.

## Rules for Extraction

### Focus on Speaking Roles
- Only include characters who have spoken dialogue (indicated by their name appearing in a line of dialogue, e.g., JOE or DR. SMITH).

### Ignore Non-Speaking Roles
- Exclude characters mentioned only in action lines, scene descriptions, or parentheticals if they do not speak (e.g., "a man runs past" or "Sarah watches from the window").

### Consolidate Variations
- Identify and merge different name variations for the same character. Use their most common or full name.
  - **Example:** JOE, JOE BAKER, and MR. BAKER should be consolidated into Joe Baker.
  - **Example:** KATE and KATHERINE (if confirmed to be the same person) should be Katherine (Kate).

### Exclude Background & Grouped Characters
- Ignore generic descriptors like COP #1, CROWD, VOICE (O.S.), NARRATOR, or DRIVER unless they are individually significant and named in the dialogue cues (e.g., OFFICER JENKINS).

### Format and Case
- Present names in proper case (e.g., Ellen Ripley, not ELLEN RIPLEY). The list should be clean, readable, and alphabetized.

## Output Format
Return the list strictly in the following XML-like format. Do not include any other commentary, analysis, or text before or after the tags.

```xml
<CharacterNames>
- Character Full Name 1
- Character Full Name 2
- Character Full Name 3
</CharacterNames>
```

## Fallback
If no identifiable speaking characters are present, return an empty list within the tags.

## Example

### Input:
```
INT. COFFEE SHOP - DAY

JOE (25), looking tired, sips his espresso. Sarah watches him from across the counter.

JOE
(long sigh)
Rough night.

SARAH
I can tell. Want a refill?

A MAN in a trench coat bursts through the door.

MAN
Everyone stay calm!

Joe spills his coffee.
```

### Output:
```xml
<CharacterNames>
- Joe
- Sarah
</CharacterNames>
```

    """

    messages=[
        ("system",charecter_prompt),
        ("human",script),
    ]
    result=model.invoke(messages)

    inside = re.search(r"<CharacterNames>(.*?)</CharacterNames>", result.content[0]['text'], re.DOTALL).group(1)
    names = [line.strip("- ").strip() for line in inside.splitlines() if line.strip()]

    return names

##### Breaking down the script to scenes

In [ ]:

def get_scenes(script,model,characters):

    scene_description_prompt=f"""
You are a **Scene-to-Concept-Art Breakdown AI**.  
Your role is to analyze a full screenplay and extract **visual moments** that can be turned into **concept art**, while maintaining **continuity** across the story.  

### Core Task
- Decompose the screenplay into **smaller, visualizable shots or frames**.  
- Each `<scene>` should contain **all key elements** needed for concept art while ensuring **consistent style across the script**.  

### Guidelines
1. **Essential Visuals**  
   - Capture: **location, time of day, lighting, characters, key objects, and mood/atmosphere**.  
   - Ignore dialogue unless it contributes to visuals.  

2. **Scene Granularity**  
   - Break long scenes into **multiple visual moments** (e.g., establishing shot, close-up, action beat).  
   - Each should stand alone as **unique concept art**.  

3. **Continuity Elements**  
   - Always specify:  
     - **<location>** (e.g., rooftop apartment, desert road).  
     - **<time>** (e.g., dawn, afternoon, midnight).  
     - **<lighting>** (e.g., warm sunlight, neon glow, candlelit).  
     - **<mood>** (e.g., ominous, hopeful, tense, romantic).  
   - Ensure characters look consistent across all scenes they appear in.  

4. **Concise but Cinematic**  
   - Keep each description **short, vivid, and cinematic** (for image generation).  
   - Focus only on **visual storytelling details**.  


### **Output Format**
You will output each scene strictly and exclusively in the following XML-like format:

```
<scene>
<description>[A complete, detailed visual description. Include character actions, environment, atmosphere, props, and the implied camera shot. Must be fully self-contained.]</description>
<location>[Specific location with architectural/environmental details]</location>
<time>[Precise time of day and weather/seasonal conditions]</time>
<lighting>[Detailed black and white lighting setup. Describe source, direction, intensity, contrast, and the mood it creates.]</lighting>
<mood>[The emotional atmosphere and visual tone of the scene]</mood>
<characters>CHARACTER_1, CHARACTER_2(only use these characters {characters})</characters>
</scene>
```
"""

    messages=[
        ("system",scene_description_prompt),
        ("human",script),
    ]
    
    resl_scene=model.invoke(messages)
    print(resl_scene)
    result=resl_scene.content[0]['text']


    scenes = re.findall(r"<scene>(.*?)</scene>", result, re.DOTALL)
    scenes = [scene.strip() for scene in scenes]

    return scenes

##### Encoding charecter images to base64

In [ ]:
import os
import base64

def encode_character_images(characters, folder="./images"):
    """
    Takes a list of character names and returns a dictionary
    with base64-encoded images.
    
    Args:
        characters (list of str): List of character names
        folder (str): Path to the folder containing images
    
    Returns:
        dict: {character_name: base64_string}
    """
    encoded_images = {}
    
    for char in characters:
        file_path = os.path.join(folder, f"{char}.jpg")
        
        # Check if file exists
        if os.path.exists(file_path):
            with open(file_path, "rb") as image_file:
                encoded_images[char] = base64.b64encode(image_file.read()).decode("utf-8")
        else:
            print(f"Warning: Image for {char} not found at {file_path}")
    
    return encoded_images

##### Creating dynamic messages for nano banana

In [ ]:
import re
from langchain.schema import HumanMessage

def create_scene_message(scene_text, encoded_images_dict):

    # Extract character names from scene
    match = re.search(r"<characters>(.*?)</characters>", scene_text, re.IGNORECASE)
    if match:
        scene_characters = [c.strip().title() for c in match.group(1).split(",")]
    else:
        scene_characters = []

    content_list = []

    # Add images only for characters present in this scene
    for char in scene_characters:
        if char in encoded_images_dict:
            content_list.append({"type": "text", "text": f"Here is an image of the character {char}:"})
            content_list.append({"type": "image_url", "image_url": f"data:image/png;base64,{encoded_images_dict[char]}"})
        else:
            print(f"Warning: No image found for character {char}")


    # image_file_path = "./style.png"

    # with open(image_file_path, "rb") as image_file:
    #     Style_enc = base64.b64encode(image_file.read()).decode("utf-8")

    # content_list.append({"type": "text", "text":'reference image'  f":"})
    # content_list.append({"type": "image_url", "image_url": f"data:image/png;base64,{Style_enc}"})



    content_list.append({
        "type": "text",
        "text": f"""

    Create a 2d black and white story board panels for the scene.

    scene:{scene_text}

"""
    })



    return HumanMessage(content=content_list)


##### Nano Banana Response to base64

In [ ]:
from langchain_core.messages import AIMessage

def _get_image_base64(response: AIMessage):
    """Extract base64 image string from response if available."""
    for block in response.content:
        if isinstance(block, dict) and block.get("image_url"):
            return block["image_url"].get("url").split(",")[-1]
    return None

##### Nano Banana activation and saving to folder

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

def generate_save(scenes,enc_char):
    os.makedirs("./output", exist_ok=True)

    nano_banana=ChatGoogleGenerativeAI(
        model='models/gemini-2.5-flash-image-preview',
        temperature=0.2
    )
    i=0
    for index, scene in enumerate(scenes):
        message=create_scene_message(scene,enc_char)

        print("starting generation for scene ",index)

        response = nano_banana.invoke(
            [message],
            generation_config=dict(response_modalities=["TEXT", "IMAGE"]),
        )

        image_base64 = _get_image_base64(response)


        if image_base64 is None:
            print(f"⚠️ No image returned for scene {index}, skipping.")
            continue
        
        output_path = f"./output/final_image{i}.png"
        i+=1
        
        with open(output_path, "wb") as f:
            f.write(base64.b64decode(image_base64))

        print(f"Image saved at {output_path}")        

##### Saving images to PDF

In [ ]:
from PIL import Image
import os

def images_to_pdf(folder_path, output_pdf):
    images = sorted(
        [f for f in os.listdir(folder_path) if f.endswith(".png")],
        key=lambda x: int(x.replace("final_image", "").replace(".png", ""))
    )

    first_image = Image.open(os.path.join(folder_path, images[0])).convert("RGB")
    
    img_list = [Image.open(os.path.join(folder_path, img)).convert("RGB") for img in images[1:]]

    first_image.save(output_pdf, save_all=True, append_images=img_list)
    print(f"PDF saved as {output_pdf}")



##### Assgning models for script breakdown

In [ ]:
from langchain_openai import ChatOpenAI

model=ChatOpenAI(model='gpt-5-2025-08-07', reasoning={"effort": "high"})
model1=ChatOpenAI(model='gpt-5-2025-08-07', reasoning={"effort": "medium"})

##### Main Run commands

In [ ]:
path=''

In [ ]:
script=pdf_text(path)
charecters=extract_charecters(script,model1)
print(charecters)

In [ ]:
import sys

for char in charecters:
    filename = f"./images/{char}.jpg"
    if os.path.exists(filename):
        print(f" Found: {filename}")
    else:
        print(f" Missing: {filename}")
        sys.exit(f"Execution stopped because {filename} is missing.")


In [ ]:
enc_char=encode_character_images(charecters)

In [ ]:
scenes=get_scenes(script,model,charecters)

In [ ]:
scenes

In [ ]:
enc_char

In [ ]:
generate_save(scenes,enc_char)

##### Saving PDF

In [ ]:
output_path=''

In [ ]:
images_to_pdf(folder_path = './output', output_pdf = output_path)